# Prophet baseline

Additive forecasting baseline. Install requirements/prophet.txt before running.

Model logic is preserved from the course project. Paths are relative to this repository; generated files go to `outputs/prophet/`. Saved outputs were cleared for publication.


In [ ]:
from pathlib import Path

# Locate the repository when launched from its root or notebooks directory.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from the repository root.")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "prophet"
for name in ("train.csv", "val.csv", "test.csv"):
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(f"Missing {DATA_DIR / name}. See data/README.md.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np

# 读取数据
train_df = pd.read_csv(str(DATA_DIR / 'train.csv'))
val_df = pd.read_csv(str(DATA_DIR / 'val.csv'))
test_df = pd.read_csv(str(DATA_DIR / 'test.csv'))

# 解析日期，生成 'start_date' 列作为时间索引
def extract_start_date(date_range_str):
    # e.g. '0001/12/27 - 0002/01/02'
    return date_range_str.split(' - ')[0]

def parse_relative_date(relative_str):
    # e.g. '0001/12/27'
    year, month, day = [int(x) for x in relative_str.split('/')]
    # 用 pd.Timestamp(0) 加上相对天数确定绝对日期（可换成任何基准或简单地直接拼接字符串）
    return pd.Timestamp(year=year+2000, month=month, day=day)

train_df['start_date'] = train_df['Relative Date'].map(extract_start_date).map(parse_relative_date)
val_df['start_date'] = val_df['Relative Date'].map(extract_start_date).map(parse_relative_date)
test_df['start_date'] = test_df['Relative Date'].map(extract_start_date).map(parse_relative_date)

# 合并训练集和验证集用作最终训练
full_train = pd.concat([train_df, val_df], axis=0)
full_train = full_train.sort_values('start_date')

In [ ]:
from prophet import Prophet

# 训练集准备
df_prophet = full_train[['start_date', 'Predict Target']].rename(columns={'start_date': 'ds', 'Predict Target': 'y'})

# 建模
model = Prophet(weekly_seasonality=True, yearly_seasonality=False, daily_seasonality=False)
model.fit(df_prophet)

# 准备预测用DataFrame
future_dates = test_df[['start_date']].rename(columns={'start_date': 'ds'})
forecast = model.predict(future_dates)

# 取得预测结果
test_pred = forecast['yhat'].values

In [ ]:
# 生成提交文件：假定要求格式与test.csv完全一致，只需添加 'Predict Target'列（如有特殊格式再调整）
result_df = test_df.copy()
result_df['Predict Target'] = test_pred
result_df = result_df[['Relative Date', 'Predict Target']]
result_df.to_csv(str(OUTPUT_DIR / 'submission.csv'), index=False)

In [ ]:
from sklearn.metrics import mean_squared_error

# 使用训练集拟合，不用val
train_only = train_df[['start_date', 'Predict Target']].rename(columns={'start_date': 'ds', 'Predict Target': 'y'})
val_dates = val_df[['start_date']].rename(columns={'start_date': 'ds'})
val_actual = val_df['Predict Target'].values

model_tmp = Prophet(weekly_seasonality=True, yearly_seasonality=False, daily_seasonality=False)
model_tmp.fit(train_only)
val_forecast = model_tmp.predict(val_dates)
val_pred = val_forecast['yhat'].values
mse = mean_squared_error(val_actual, val_pred)
print('Validation MSE:', mse)